# 06C — Source Pixel Diagnostic (FIXED)

Run this **after 06B and before 07**.

It specifically diagnoses the four 100%-missing station–feature pairs found in 06B:

- `CL503 → CDR`
- `CL509 → CCS`
- `CL509 → PDIR`
- `CL509 → Distance_Sea`

For each suspect source, it checks the exact native source pixel and its surrounding 5×5 neighborhood. It does **not** fill or modify any data.

For dynamic precipitation products, January, August and December are checked for every year from 2017–2022 so we can distinguish a persistent source-coverage problem from an isolated pixel issue.


In [2]:
# ============================================================
# 06C — SOURCE PIXEL DIAGNOSTIC (FULL FIXED)
# Diagnose 100%-missing station-feature pairs
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import rasterio


# ============================================================
# 1. FIND PROJECT ROOT
# ============================================================

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. "
        "Run this notebook from inside the repository."
    )


PROJECT_ROOT = find_project_root()

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

OUT_DIR = (
    PROCESSED_DIR
    / "source_pixel_diagnostic"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("PROJECT_ROOT =", PROJECT_ROOT)


# ============================================================
# 2. LOAD NOTEBOOK 06 OUTPUT
# ============================================================

SAMPLES_PATH = (
    PROCESSED_DIR
    / "station_samples_native_tidy.csv"
)

if not SAMPLES_PATH.exists():
    raise FileNotFoundError(
        "station_samples_native_tidy.csv not found.\n"
        "Run 06_Station_Value_Extraction_NATIVE_FIXED_v2.ipynb first."
    )


SAMPLES = pd.read_csv(
    SAMPLES_PATH
)


print(
    "Samples:",
    SAMPLES.shape
)


print(
    "Stations:",
    SAMPLES["station_id"].nunique()
)


# ============================================================
# 3. PRECIPITATION FOLDERS
# ============================================================

PRECIP_FOLDERS = {

    "CCS":
        "CCS",

    "PDIR":
        "PDIR",

    "CDR":
        "CDR",

}


# ============================================================
# 4. HELPER — PARSE YEAR/MONTH
# ============================================================

def parse_ym(name):

    stem = Path(name).stem

    patterns = [

        r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",

        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"

    ]

    for pat in patterns:

        m = re.search(
            pat,
            stem
        )

        if m:

            return (
                int(m.group(1)),
                int(m.group(2))
            )

    return None


# ============================================================
# 5. HELPER — LIST RASTERS
# ============================================================

def rasters(folder):

    if not folder.exists():

        return []

    return sorted([

        *folder.rglob("*.tif"),

        *folder.rglob("*.tiff")

    ])


# ============================================================
# 6. HELPER — MONTHLY MAP
# ============================================================

def monthly_map(folder):

    d = {}

    for p in rasters(folder):

        ym = parse_ym(
            p.name
        )

        if ym is None:
            continue

        if ym in d:

            raise ValueError(

                f"Duplicate raster for {folder.name} {ym}:\n"
                f"{d[ym]}\n"
                f"{p}"
            )

        d[ym] = p

    return d


# ============================================================
# 7. BUILD PRECIPITATION MAPS
# ============================================================

maps = {}


for feature, folder_name in (
    PRECIP_FOLDERS.items()
):

    folder = (

        RAW_DIR
        / "precipitation"
        / folder_name

    )

    if not folder.exists():

        raise FileNotFoundError(
            f"Missing folder:\n{folder}"
        )

    maps[
        feature
    ] = monthly_map(
        folder
    )

    print(
        feature,
        "parsed =",
        len(
            maps[
                feature
            ]
        )
    )


# ============================================================
# 8. FIND CANONICAL DISTANCE-TO-SEA
# ============================================================

distance_folder = (

    RAW_DIR
    / "predictors"
    / "Distance_Sea"

)


distance_files = rasters(
    distance_folder
)


if not distance_files:

    raise FileNotFoundError(
        "No Distance_Sea raster found."
    )


distance_lookup = {

    p.name.lower(): p

    for p in distance_files
}


DISTANCE_PATH = (
    distance_lookup.get(
        "distance_sea.tif"
    )
)


if DISTANCE_PATH is None:

    clean = [

        p for p in distance_files

        if not any(

            x in p.stem.lower()

            for x in [

                "clip",
                "tmp",
                "temp",
                "aligned",
                "resampl"

            ]
        )

    ]

    if len(
        clean
    ) == 1:

        DISTANCE_PATH = (
            clean[0]
        )

    else:

        raise ValueError(

            "Could not uniquely identify "
            "canonical Distance_Sea raster."
        )


print(
    "Distance_Sea =",
    DISTANCE_PATH
)


# ============================================================
# 9. TARGET PROBLEMS FROM 06B
# ============================================================

TARGETS = [

    (
        "CL503",
        "CDR"
    ),

    (
        "CL509",
        "CCS"
    ),

    (
        "CL509",
        "PDIR"
    ),

    (
        "CL509",
        "Distance_Sea"
    ),

]


# ============================================================
# 10. STATION COORDINATES
# ============================================================

coords = (

    SAMPLES
    .groupby(
        "station_id"
    )
    .agg(

        latitude=(
            "latitude",
            "median"
        ),

        longitude=(
            "longitude",
            "median"
        )

    )

)


print(
    "\n=========================================="
)

print(
    "TARGET STATIONS"
)

print(
    "=========================================="
)


display(

    coords.loc[
        [
            "CL503",
            "CL509"
        ]
    ]

)


# ============================================================
# 11. HELPER — GEOGRAPHIC BOUNDS
# ============================================================

def is_geographic_bounds(
    b
):

    return (

        -180 <= b.left <= 180

        and

        -180 <= b.right <= 180

        and

        -90 <= b.bottom <= 90

        and

        -90 <= b.top <= 90

    )


# ============================================================
# 12. INSPECT EXACT PIXEL + 5x5 NEIGHBORHOOD
# ============================================================

def inspect_pixel(
    path,
    lon,
    lat,
    radius=2
):


    with rasterio.open(
        path
    ) as src:


        b = src.bounds


        result = {

            "source":
                str(
                    path
                ),

            "crs":
                str(
                    src.crs
                ),

            "width":
                src.width,

            "height":
                src.height,

            "res_x":
                src.res[0],

            "res_y":
                src.res[1],

            "nodata":
                src.nodata,

            "left":
                b.left,

            "bottom":
                b.bottom,

            "right":
                b.right,

            "top":
                b.top,

            "bounds_geographic":
                is_geographic_bounds(
                    b
                ),

        }


        # ====================================================
        # REQUIRE GEOGRAPHIC-LOOKING BOUNDS
        # ====================================================

        if not is_geographic_bounds(
            b
        ):

            result[
                "diagnostic"
            ] = (
                "NON_GEOGRAPHIC_BOUNDS_REQUIRES_CRS_REVIEW"
            )

            return (
                result,
                None
            )


        x = float(
            lon
        )

        y = float(
            lat
        )


        # ====================================================
        # CHECK STATION INSIDE RASTER BOUNDS
        # ====================================================

        inside = (

            b.left <= x <= b.right

            and

            b.bottom <= y <= b.top

        )


        result[
            "station_inside_bounds"
        ] = inside


        if not inside:

            result[
                "diagnostic"
            ] = (
                "OUTSIDE_RASTER_BOUNDS"
            )

            return (
                result,
                None
            )


        # ====================================================
        # GET EXACT ROW / COLUMN
        # ====================================================

        try:

            row, col = src.index(
                x,
                y
            )

        except Exception:

            result[
                "diagnostic"
            ] = (
                "INDEX_FAILED"
            )

            return (
                result,
                None
            )


        result[
            "row"
        ] = row

        result[
            "col"
        ] = col


        if (

            row < 0

            or

            row >= src.height

            or

            col < 0

            or

            col >= src.width

        ):

            result[
                "diagnostic"
            ] = (
                "INDEX_OUTSIDE"
            )

            return (
                result,
                None
            )


        # ====================================================
        # READ EXACT PIXEL
        # ====================================================

        exact = src.read(

            1,

            window=(

                (
                    row,
                    row + 1
                ),

                (
                    col,
                    col + 1
                )

            ),

            masked=True

        )


        # ====================================================
        # EXACT VALUE
        # ====================================================

        if (
            exact.size == 0
        ):

            exact_value = (
                np.nan
            )


        elif np.ma.is_masked(
            exact[
                0,
                0
            ]
        ):

            exact_value = (
                np.nan
            )


        else:

            exact_value = float(

                exact[
                    0,
                    0
                ]

            )


            if not np.isfinite(
                exact_value
            ):

                exact_value = (
                    np.nan
                )


            if (
                src.nodata is not None

                and

                np.isclose(
                    exact_value,
                    src.nodata
                )
            ):

                exact_value = (
                    np.nan
                )


        result[
            "exact_value"
        ] = exact_value


        result[
            "exact_is_missing"
        ] = bool(
            np.isnan(
                exact_value
            )
        )


        # ====================================================
        # BUILD 5x5 NEIGHBORHOOD
        # ====================================================

        r0 = max(
            0,
            row - radius
        )

        r1 = min(
            src.height,
            row + radius + 1
        )

        c0 = max(
            0,
            col - radius
        )

        c1 = min(
            src.width,
            col + radius + 1
        )


        arr = src.read(

            1,

            window=(

                (
                    r0,
                    r1
                ),

                (
                    c0,
                    c1
                )

            ),

            masked=True

        )


        # ====================================================
        # IMPORTANT FIX:
        # Convert integer masked array to float BEFORE NaN fill
        # ====================================================

        arr_float = arr.astype(
            "float64"
        )


        filled = np.asarray(

            arr_float.filled(
                np.nan
            ),

            dtype=float

        )


        valid = np.isfinite(
            filled
        )


        result[
            "neighbor_total_pixels"
        ] = (
            filled.size
        )


        result[
            "neighbor_valid_pixels"
        ] = int(
            valid.sum()
        )


        # ====================================================
        # NEIGHBOR STATISTICS
        # ====================================================

        if valid.any():

            result[
                "neighbor_min"
            ] = float(
                np.nanmin(
                    filled
                )
            )


            result[
                "neighbor_max"
            ] = float(
                np.nanmax(
                    filled
                )
            )


            result[
                "neighbor_mean"
            ] = float(
                np.nanmean(
                    filled
                )
            )


        else:

            result[
                "neighbor_min"
            ] = np.nan

            result[
                "neighbor_max"
            ] = np.nan

            result[
                "neighbor_mean"
            ] = np.nan


        # ====================================================
        # DIAGNOSTIC CLASS
        # ====================================================

        if np.isnan(
            exact_value
        ):

            if valid.any():

                result[
                    "diagnostic"
                ] = (
                    "EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID"
                )


            else:

                result[
                    "diagnostic"
                ] = (
                    "LOCAL_NODATA_BLOCK"
                )


        else:

            result[
                "diagnostic"
            ] = (
                "EXACT_PIXEL_VALID"
            )


        # ====================================================
        # SAVE NEIGHBORHOOD TABLE
        # ====================================================

        neighborhood = pd.DataFrame(
            filled
        )


        neighborhood.index = [

            f"r{r}"

            for r in range(
                r0,
                r1
            )

        ]


        neighborhood.columns = [

            f"c{c}"

            for c in range(
                c0,
                c1
            )

        ]


        return (
            result,
            neighborhood
        )


# ============================================================
# 13. RUN DIAGNOSTIC
# ============================================================

rows = []

neighborhood_exports = []


for station, feature in TARGETS:


    lat = float(

        coords.loc[
            station,
            "latitude"
        ]

    )


    lon = float(

        coords.loc[
            station,
            "longitude"
        ]

    )


    # ========================================================
    # STATIC DISTANCE-TO-SEA
    # ========================================================

    if feature == "Distance_Sea":

        source_items = [

            (
                "static",
                DISTANCE_PATH
            )

        ]


    # ========================================================
    # DYNAMIC PRECIPITATION
    # ========================================================

    else:

        source_items = []


        for year in range(
            2017,
            2023
        ):


            for month in [
                1,
                8,
                12
            ]:


                p = maps[
                    feature
                ].get(

                    (
                        year,
                        month
                    )

                )


                if p is None:

                    raise FileNotFoundError(

                        f"Missing {feature} raster "
                        f"for {year}-{month:02d}"

                    )


                source_items.append(

                    (
                        f"{year}-{month:02d}",
                        p
                    )

                )


    # ========================================================
    # INSPECT EACH SOURCE
    # ========================================================

    for period, path in source_items:


        info, neighborhood = inspect_pixel(

            path,
            lon,
            lat,
            radius=2

        )


        rec = {

            "station_id":
                station,

            "feature":
                feature,

            "period":
                period,

            "latitude":
                lat,

            "longitude":
                lon,

            **info

        }


        rows.append(
            rec
        )


        # ====================================================
        # SAVE 5x5 NEIGHBORHOOD
        # ====================================================

        if neighborhood is not None:


            safe_period = period.replace(
                "-",
                "_"
            )


            fn = (

                f"{station}_"
                f"{feature}_"
                f"{safe_period}_"
                f"neighborhood_5x5.csv"

            )


            neighborhood.to_csv(

                OUT_DIR
                / fn

            )


            neighborhood_exports.append(
                fn
            )


# ============================================================
# 14. DETAIL RESULTS
# ============================================================

diagnostic = pd.DataFrame(
    rows
)


print(
    "\n=========================================="
)

print(
    "SOURCE PIXEL DIAGNOSTIC RESULTS"
)

print(
    "=========================================="
)


result_cols = [

    "station_id",

    "feature",

    "period",

    "station_inside_bounds",

    "exact_value",

    "exact_is_missing",

    "neighbor_valid_pixels",

    "neighbor_min",

    "neighbor_max",

    "neighbor_mean",

    "diagnostic"

]


display(

    diagnostic[
        result_cols
    ]

)


# ============================================================
# 15. SUMMARY BY STATION × FEATURE
# ============================================================

summary = (

    diagnostic

    .groupby(

        [
            "station_id",
            "feature"
        ],

        dropna=False

    )

    .agg(

        checks=(
            "period",
            "size"
        ),

        inside_bounds=(

            "station_inside_bounds",

            lambda x:
                int(
                    x.fillna(
                        False
                    ).sum()
                )

        ),

        exact_valid=(

            "exact_is_missing",

            lambda x:
                int(
                    (
                        ~x.fillna(
                            True
                        )
                    ).sum()
                )

        ),

        exact_missing=(

            "exact_is_missing",

            lambda x:
                int(
                    x.fillna(
                        True
                    ).sum()
                )

        ),

        checks_with_valid_neighbors=(

            "neighbor_valid_pixels",

            lambda x:
                int(
                    (
                        x.fillna(
                            0
                        )
                        > 0
                    ).sum()
                )

        ),

        unique_diagnostics=(

            "diagnostic",

            lambda x:
                " | ".join(

                    sorted(

                        set(

                            x.dropna()
                            .astype(
                                str
                            )

                        )

                    )

                )

        )

    )

    .reset_index()

)


print(
    "\n=========================================="
)

print(
    "SUMMARY BY STATION × FEATURE"
)

print(
    "=========================================="
)


display(
    summary
)


# ============================================================
# 16. INTERPRETATION FLAGS
# ============================================================

def classify(
    r
):

    txt = str(
        r[
            "unique_diagnostics"
        ]
    )


    if (
        "OUTSIDE_RASTER_BOUNDS"
        in txt
    ):

        return (
            "SOURCE_EXTENT_PROBLEM"
        )


    if (

        r[
            "exact_missing"
        ]
        ==
        r[
            "checks"
        ]

        and

        r[
            "checks_with_valid_neighbors"
        ]
        ==
        r[
            "checks"
        ]

    ):

        return (
            "PIXEL_HOLE_NEIGHBORS_VALID"
        )


    if (

        r[
            "exact_missing"
        ]
        ==
        r[
            "checks"
        ]

        and

        r[
            "checks_with_valid_neighbors"
        ]
        ==
        0

    ):

        return (
            "LOCAL_NODATA_REGION"
        )


    if (
        r[
            "exact_valid"
        ]
        > 0
    ):

        return (
            "MIXED_OR_UNEXPECTED"
        )


    return (
        "REVIEW"
    )


summary[
    "interpretation_flag"
] = summary.apply(

    classify,

    axis=1

)


print(
    "\n=========================================="
)

print(
    "FINAL DIAGNOSTIC FLAGS"
)

print(
    "=========================================="
)


display(

    summary[
        [
            "station_id",
            "feature",
            "checks",
            "inside_bounds",
            "exact_valid",
            "exact_missing",
            "checks_with_valid_neighbors",
            "interpretation_flag"
        ]
    ]

)


# ============================================================
# 17. SAVE OUTPUTS
# ============================================================

detail_path = (

    OUT_DIR
    / "source_pixel_diagnostic_detail.csv"

)


summary_path = (

    OUT_DIR
    / "source_pixel_diagnostic_summary.csv"

)


diagnostic.to_csv(

    detail_path,

    index=False

)


summary.to_csv(

    summary_path,

    index=False

)


# ============================================================
# 18. FINAL
# ============================================================

print(
    "\n=========================================="
)

print(
    "06C SOURCE PIXEL DIAGNOSTIC COMPLETE"
)

print(
    "=========================================="
)


print(
    "Saved detail:",
    detail_path
)


print(
    "Saved summary:",
    summary_path
)


print(
    "Neighborhood CSVs:",
    len(
        neighborhood_exports
    )
)


print(
    "\nIMPORTANT:"
)

print(
    "This notebook only diagnoses source pixels."
)

print(
    "It does NOT fill, interpolate, clamp, "
    "or modify any predictor."
)

PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh
Samples: (432, 19)
Stations: 6
CCS parsed = 72
PDIR parsed = 72
CDR parsed = 72
Distance_Sea = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\Distance_Sea\Distance_Sea.tif

TARGET STATIONS


,latitude,longitude
station_id,,
CL503,22.6012,89.5195
CL509,22.6887,89.3088



SOURCE PIXEL DIAGNOSTIC RESULTS


,station_id,feature,period,station_inside_bounds,exact_value,exact_is_missing,neighbor_valid_pixels,neighbor_min,neighbor_max,neighbor_mean,diagnostic
0,CL503,CDR,2017-01,True,NaN,True,5,4.062212,6.653240,5.413066,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
1,CL503,CDR,2017-08,True,NaN,True,5,393.043243,460.899658,423.624451,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
2,CL503,CDR,2017-12,True,NaN,True,5,53.176075,65.222023,59.848808,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
3,CL503,CDR,2018-01,True,NaN,True,5,5.215755,10.399554,7.994988,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
4,CL503,CDR,2018-08,True,NaN,True,5,207.805740,266.552490,234.465527,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
5,CL503,CDR,2018-12,True,NaN,True,5,8.691934,17.413019,12.976896,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
6,CL503,CDR,2019-01,True,NaN,True,5,0.000000,0.833826,0.166765,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
7,CL503,CDR,2019-08,True,NaN,True,5,304.240967,351.312653,330.527466,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
8,CL503,CDR,2019-12,True,NaN,True,5,6.090306,11.395324,7.795716,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
9,CL503,CDR,2020-01,True,NaN,True,5,23.724394,31.742348,27.336448,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID



SUMMARY BY STATION × FEATURE


,station_id,feature,checks,inside_bounds,exact_valid,exact_missing,checks_with_valid_neighbors,unique_diagnostics
0,CL503,CDR,18,18,0,18,18,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
1,CL509,CCS,18,18,0,18,18,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
2,CL509,Distance_Sea,1,1,0,1,1,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID
3,CL509,PDIR,18,18,0,18,18,EXACT_PIXEL_NODATA_BUT_NEIGHBORS_VALID



FINAL DIAGNOSTIC FLAGS


,station_id,feature,checks,inside_bounds,exact_valid,exact_missing,checks_with_valid_neighbors,interpretation_flag
0,CL503,CDR,18,18,0,18,18,PIXEL_HOLE_NEIGHBORS_VALID
1,CL509,CCS,18,18,0,18,18,PIXEL_HOLE_NEIGHBORS_VALID
2,CL509,Distance_Sea,1,1,0,1,1,PIXEL_HOLE_NEIGHBORS_VALID
3,CL509,PDIR,18,18,0,18,18,PIXEL_HOLE_NEIGHBORS_VALID



06C SOURCE PIXEL DIAGNOSTIC COMPLETE
Saved detail: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\source_pixel_diagnostic\source_pixel_diagnostic_detail.csv
Saved summary: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\source_pixel_diagnostic\source_pixel_diagnostic_summary.csv
Neighborhood CSVs: 55

IMPORTANT:
This notebook only diagnoses source pixels.
It does NOT fill, interpolate, clamp, or modify any predictor.
